In [ ]:
```json
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ODI to Databricks Migration: SILOS_SIL_INVENTORYPRODUCTDIMENSION\n",
        "**Source:** PRXBI_DW Oracle schema\n",
        "**Target:** workspace.prxbi_dw (Databricks Delta Lake)\n",
        "**Description:** Inventory Product Dimension ETL with incremental update/insert logic\n",
        "**Conversion Date:** 2024\n",
        "\n",
        "## Overview\n",
        "This notebook migrates an ODI IKM BIAPPS Oracle Incremental Update session to Databricks Spark SQL.\n",
        "Key transformations:\n",
        "- Oracle schema PRXBI_DW → workspace.prxbi_dw\n",
        "- Flow table I$_3260538_1 → i_3260538_1_flow\n",
        "- Error table E$_3260538_1 → e_3260538_1\n",
        "- Staging table C$ → c_inventory_product_stg\n",
        "- Oracle functions (NVL, SYSDATE, TO_TIMESTAMP, DECODE) converted to Spark equivalents\n",
        "- UPDATE + INSERT logic converted to single MERGE statement\n",
        "- Dimension lookups integrated inline\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Create ETL parameter widgets\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_USAGE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"SOURCE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"TARGET_CODE\", \"\")\n",
        "dbutils.widgets.text(\"LOW_DATE\", \"1900-01-01 00:00:00\")\n",
        "dbutils.widgets.text(\"IS_INCREMENTAL\", \"Y\")\n",
        "dbutils.widgets.text(\"PRUNE_DAYS\", \"0\")\n",
        "dbutils.widgets.text(\"EXECUTION_ID\", \"\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## ETL Parameters\n",
        "Display all parameter values for validation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Display parameter values\n",
        "params = {\n",
        "    'DATASOURCE_NUM_ID': dbutils.widgets.get('DATASOURCE_NUM_ID'),\n",
        "    'ETL_PROC_WID': dbutils.widgets.get('ETL_PROC_WID'),\n",
        "    'ETL_USAGE_CODE': dbutils.widgets.get('ETL_USAGE_CODE'),\n",
        "    'SOURCE_CODE': dbutils.widgets.get('SOURCE_CODE'),\n",
        "    'TARGET_CODE': dbutils.widgets.get('TARGET_CODE'),\n",
        "    'LOW_DATE': dbutils.widgets.get('LOW_DATE'),\n",
        "    'IS_INCREMENTAL': dbutils.widgets.get('IS_INCREMENTAL'),\n",
        "    'PRUNE_DAYS': dbutils.widgets.get('PRUNE_DAYS'),\n",
        "    'EXECUTION_ID': dbutils.widgets.get('EXECUTION_ID')\n",
        "}\n",
        "display(params)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## SCEN_TASK_NO {1}: Validate ETL Load Configuration\n",
        "Check if package exists in ETL_LOAD_DATES table"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "SELECT CASE\n",
        "    WHEN COUNT(*) > 0 THEN 'Y'\n",
        "    ELSE 'N'\n",
        "END AS package_exists\n",
        "FROM workspace.prxbi_dw.w_etl_load_dates\n",
        "WHERE package_name = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "  AND (datasource_num_id = ${DATASOURCE_NUM_ID}\n",
        "    OR datasource_num_id = 999)\n",
        "  AND etl_usage_code = '${ETL_USAGE_CODE}'\n",
        "  AND committed = '1'"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## SCEN_TASK_NO {2}: Merge Category Updates\n",
        "Update inventory product with category lookups from temporary staging"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- This step merges category updates from the source\n",
        "-- Note: Actual merge deferred until full flow table population\n",
        "-- SCEN_TASK_NO {2} placeholder - full implementation in main MERGE statement below"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Error and Check Tables\n",
        "Create persistent error logging and audit tables"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},